# 04 — Stability and sensitivity analyses (Layer 5)

Inputs: the preprocessed data (`01_preprocessing.ipynb`) and the main results
(`02_features_group_comparison.ipynb` and `03_condition_dependence.ipynb`), computed with
seed 42. Feature extraction and Layers 1–3 use the same functions as the main analysis
(`src/eeg_features.py` and `src/condition_tests.py`).

1. **Window selection.** Features and Layers 1–3 are recomputed with `N_SEEDS` seeds, always
   including seed 42 (the main analysis, read from `RESULTS_DIR`). Each seed changes both the
   selection of 2-s windows and the selection of 10-s blocks; the sample, `n_target` and
   `n_long_target` do not change. Stability is measured by the Jaccard index between the
   sets of significant features of each seed and those of seed 42, the detection rate of
   each feature and the dispersion of Cliff's delta across seeds, by feature and by family.
2. **Amplitude scale.** Features and Layers 1–3 are recomputed with seed 42 after dividing
   each record (subject and condition) by one global RMS factor, computed over all windows,
   channels and samples, which preserves the differences between channels.
3. **Participant HC_4.** Its TASK record was attributed from metadata
   (`00_duplicate_audit.ipynb`), so Layers 1–3 are repeated without this participant, with the
   features of the main analysis.

Variation across seeds measures the dependence on the selected windows, not uncertainty
over participants: all seeds use the same participants, so they do not separate a group
effect from the recording-batch effect.

**Outputs (in `RESULTS_DIR/layer5`):**

| File | Content |
|---|---|
| `seed_<N>/` | Features, Layer 1–3 results and run information of each seed |
| `seed_counts.csv`, `seed_counts_summary.csv` | Counts per seed (significant features, interactions, equivalences) and their distribution |
| `jaccard_vs_seed42.csv` | Jaccard index of each set against seed 42 |
| `feature_stability.csv`, `family_stability.csv` | Detection rate and Cliff's delta across seeds, by feature and by family |
| `domain_counts.csv` | Significant features by domain, condition and seed |
| `rms_global/`, `rms_feature_comparison.csv`, `rms_summary.csv` | Amplitude-normalized run and comparison with the main analysis |
| `without_HC_4/`, `without_HC_4_summary.csv` | Layers 1–3 without HC_4 |

## Setup

In [ ]:
# Google Colab only: install dependencies and mount Google Drive.
import sys
if 'google.colab' in sys.modules:
    !pip install -q PyWavelets statsmodels
    from google.colab import drive
    drive.mount('/content/drive')

In [ ]:
import os
import sys

REPO_DIR = '..'   
sys.path.insert(0, os.path.join(REPO_DIR, 'src'))

import glob
import hashlib
import json
import time
import warnings

import joblib
import numpy as np
import pandas as pd
import scipy
import statsmodels
from joblib import Parallel, delayed
from scipy.stats import spearmanr

import condition_tests as C
import eeg_features as F
from eeg_features import CONDITIONS, domain_of, family_of

warnings.filterwarnings('ignore')
print(f'Python {sys.version.split()[0]}')
for module in (np, scipy, pd, statsmodels, joblib):
    print(f'{module.__name__:12s} {module.__version__}')

## Configuration

In [ ]:
PREPROCESSED_DIR = 'path/to/preprocessed'   # OUTPUT_DIR of 01_preprocessing.ipynb
RESULTS_DIR = 'path/to/results'             # main results (notebooks 02 and 03)
LAYER5_DIR = os.path.join(RESULTS_DIR, 'layer5')

N_SEEDS = 30                  # seeds, including seed 42 (use 3 for a quick check)
SEED_SEQUENCE_ENTROPY = 20260930
N_JOBS = min(4, os.cpu_count() or 1)   # seeds run in parallel; reduce if memory runs out
EXCLUDED_SUBJECT = 'HC_4'     # TASK record attributed from metadata

MAIN_SEED = F.SEED
state = np.random.SeedSequence(SEED_SEQUENCE_ENTROPY).generate_state(max(N_SEEDS - 1, 0))
SEEDS = ([MAIN_SEED] + [int(s) for s in state if int(s) != MAIN_SEED])[:N_SEEDS]
os.makedirs(LAYER5_DIR, exist_ok=True)
pd.DataFrame({'iteration': range(len(SEEDS)), 'seed': SEEDS}).to_csv(os.path.join(LAYER5_DIR, 'seeds.csv'), index=False)
print(f'{len(SEEDS)} seeds: {SEEDS}')

## Sample of the main analysis

The analytic sample, `n_target` and `n_long_target` are recomputed and checked against the
sampling logs of the main analysis, so that every run below differs from it only in the seed,
the amplitude normalization or the excluded participant.

In [ ]:
F.check_channel_order(PREPROCESSED_DIR)
sample, n_target, n_long_target, _, _ = F.select_sample(PREPROCESSED_DIR, verbose=False)

for cond in CONDITIONS:
    log = pd.read_csv(os.path.join(RESULTS_DIR, f'sampling_log_{cond}.csv'))
    assert set(log.subject) == sample, f'{cond}: sample differs from the main analysis'
    assert (log.global_seed == MAIN_SEED).all() and (log.amplitude_normalization == 'none').all()
    assert (log.n_windows_used == np.minimum(log.n_windows_available, n_target)).all()
    assert (log.n_blocks_used == n_long_target).all()
FEATURES = list(pd.read_csv(os.path.join(RESULTS_DIR, 'features_EC.csv'), nrows=0).columns[2:])
n_hc = sum(s.startswith('HC') for s in sample)
print(f'Sample: {len(sample)} ({n_hc} HC, {len(sample) - n_hc} MDD) | n_target = {n_target} | '
      f'n_long_target = {n_long_target} | {len(FEATURES)} features | main analysis checked')

## Functions

`run_analysis` recomputes the features of the three conditions and Layers 1–3 for one seed
and one amplitude normalization. Each run is saved in its own folder with a `run_info.json`
file, written last, that records the seed, the sample, the targets, all parameters and a
SHA-256 hash of the source files in `src/`. A folder is reused only if its `run_info.json`
is identical, so an interrupted run can be resumed.

In [ ]:
def source_hash():
    h = hashlib.sha256()
    for p in sorted(glob.glob(os.path.join(REPO_DIR, 'src', '*.py'))):
        h.update(os.path.basename(p).encode())
        h.update(open(p, 'rb').read())
    return h.hexdigest()


def run_info(seed, normalization):
    return {'seed': int(seed), 'amplitude_normalization': normalization or 'none',
            'subjects': sorted(sample), 'n_target': int(n_target), 'n_long_target': int(n_long_target),
            'n_permutations': int(C.PERM_N), 'source_sha256': source_hash(),
            'parameters_features': F.parameters(), 'parameters_tests': C.parameters()}


def is_done(folder, info):
    p = os.path.join(folder, 'run_info.json')
    if not os.path.exists(p):
        return False
    with open(p) as fh:
        return json.load(fh) == json.loads(json.dumps(info))


def run_analysis(seed, normalization, folder, info):
    # Features (no bootstrap CI) and Layers 1-3 for one seed; files as in the main analysis.
    os.makedirs(folder, exist_ok=True)
    features, stats = {}, {}
    for cond in CONDITIONS:
        stats[cond], features[cond] = F.run_condition(
            PREPROCESSED_DIR, cond, sample, n_target, n_long_target, seed=seed, ci=False,
            amplitude_normalization=normalization, save_dir=folder, verbose=False)
    F.layer1_summary(stats).to_csv(os.path.join(folder, 'layer1_summary.csv'), index=False)
    long_data = C.long_from_wide(features)
    layer2 = C.run_layer2(long_data, FEATURES, n_perm=info['n_permutations'], save_dir=folder, verbose=False)
    C.run_layer3(layer2, save_dir=folder)
    with open(os.path.join(folder, 'run_info.json'), 'w') as fh:
        json.dump(info, fh, indent=1)
    return folder


def run_many(jobs):
    # jobs: list of (seed, normalization, folder); folders with identical run_info.json are reused.
    todo = []
    for seed, norm, folder in jobs:
        info = run_info(seed, norm)
        if not is_done(folder, info):
            todo.append((seed, norm, folder, info))
    print(f'{len(jobs)} runs: {len(jobs) - len(todo)} reused, {len(todo)} to compute with {N_JOBS} parallel jobs')
    t0 = time.time()
    with joblib.parallel_config(backend='loky', inner_max_num_threads=1):
        Parallel(n_jobs=N_JOBS, verbose=10)(delayed(run_analysis)(*job) for job in todo)
    print(f'Done in {(time.time() - t0) / 60:.1f} min')


def read_run(folder):
    # Layer 1 (significance and Cliff's delta per condition) and Layers 2-3 of one run.
    out = None
    for cond in CONDITIONS:
        s = pd.read_csv(os.path.join(folder, f'stats_{cond}.csv'), float_precision='round_trip')
        s = s[['feature', 'significant', 'cliffs_delta']].rename(
            columns={'significant': f'sig_{cond}', 'cliffs_delta': f'delta_{cond}'})
        out = s if out is None else out.merge(s, on='feature')
    l3 = pd.read_csv(os.path.join(folder, 'layer3_equivalence.csv'), float_precision='round_trip')
    out = out.merge(l3[['feature', 'model_ok', 'p_interaction_perm', 'p_interaction_fdr', 'interaction_significant',
                        'p_equivalence_fdr', 'equivalence_significant', 'formal_condition_status']], on='feature')
    out['sig_any'] = out[[f'sig_{c}' for c in CONDITIONS]].any(axis=1)
    out['sig_all'] = out[[f'sig_{c}' for c in CONDITIONS]].all(axis=1)
    out.insert(1, 'domain', out.feature.map(domain_of))
    out.insert(2, 'family', out.feature.map(family_of))
    return out


def counts(r):
    row = {f'n_sig_{c}': int(r[f'sig_{c}'].sum()) for c in CONDITIONS}
    row.update(n_sig_any=int(r.sig_any.sum()), n_sig_all=int(r.sig_all.sum()),
               n_interaction_fdr=int(r.interaction_significant.sum()),
               n_interaction_nominal=int((r.p_interaction_perm < 0.05).sum()),
               min_q_interaction=float(r.p_interaction_fdr.min()),
               n_equivalent_fdr=int(r.equivalence_significant.sum()),
               min_q_equivalence=float(r.p_equivalence_fdr.min()),
               n_models_converged=int(r.model_ok.sum()))
    for status in ('condition-dependent', 'condition-equivalent', 'inconclusive'):
        row[f'n_{status}'] = int(r.formal_condition_status.eq(status).sum())
    return row


SETS = [f'sig_{c}' for c in CONDITIONS] + ['sig_any', 'sig_all', 'interaction_significant', 'equivalence_significant']


def jaccard(a, b):
    # NaN when both sets are empty.
    union = a | b
    return len(a & b) / len(union) if union else np.nan


def compare_sets(ref, cur):
    rows = []
    for s in SETS:
        a, b = set(ref.feature[ref[s]]), set(cur.feature[cur[s]])
        rows.append({'set': s, 'jaccard': jaccard(a, b), 'both_empty': not (a | b),
                     'n_reference': len(a), 'n_current': len(b), 'n_intersection': len(a & b)})
    return pd.DataFrame(rows)

## 1. Stability across window selections

Seed 42 is the main analysis and is read from `RESULTS_DIR`; the other seeds are computed.
Each run takes about as long as notebook 02 without the bootstrap CIs plus Layer 2.

In [ ]:
seed_dir = lambda s: RESULTS_DIR if s == MAIN_SEED else os.path.join(LAYER5_DIR, f'seed_{s}')
run_many([(s, None, seed_dir(s)) for s in SEEDS if s != MAIN_SEED])
runs = {s: read_run(seed_dir(s)) for s in SEEDS}
reference = runs[MAIN_SEED]

In [ ]:
# Counts per seed and their distribution across seeds
seed_counts = pd.DataFrame([{'seed': s, **counts(r)} for s, r in runs.items()])
seed_counts.to_csv(os.path.join(LAYER5_DIR, 'seed_counts.csv'), index=False)
metrics = seed_counts.columns.drop('seed')
counts_summary = pd.DataFrame({
    'seed_42': seed_counts.set_index('seed').loc[MAIN_SEED, metrics],
    'median': seed_counts[metrics].median(), 'p2_5': seed_counts[metrics].quantile(0.025),
    'p97_5': seed_counts[metrics].quantile(0.975), 'min': seed_counts[metrics].min(),
    'max': seed_counts[metrics].max()})
counts_summary.to_csv(os.path.join(LAYER5_DIR, 'seed_counts_summary.csv'))
print(f'{len(SEEDS)} seeds')
print(counts_summary.round(3).to_string())

# Jaccard index against seed 42
jac = pd.concat([compare_sets(reference, r).assign(seed=s) for s, r in runs.items() if s != MAIN_SEED],
                ignore_index=True)
jac.to_csv(os.path.join(LAYER5_DIR, 'jaccard_vs_seed42.csv'), index=False)
print('\nJaccard index against seed 42 (NaN = both sets empty):')
print(jac.groupby('set', sort=False).agg(median=('jaccard', 'median'), min=('jaccard', 'min'),
                                         max=('jaccard', 'max'), runs_both_empty=('both_empty', 'sum'))
      .round(3).to_string())

In [ ]:
# Detection rate and Cliff's delta across seeds, by feature and by family
allruns = pd.concat([r.assign(seed=s) for s, r in runs.items()], ignore_index=True)
g = allruns.groupby('feature')
fs = reference[['feature', 'domain', 'family']].set_index('feature')
for c in CONDITIONS:
    fs[f'detection_rate_{c}'] = g[f'sig_{c}'].mean()
    fs[f'delta_seed42_{c}'] = reference.set_index('feature')[f'delta_{c}']
    fs[f'delta_mean_{c}'] = g[f'delta_{c}'].mean()
    fs[f'delta_sd_{c}'] = g[f'delta_{c}'].std(ddof=1)
    fs[f'delta_p2_5_{c}'] = g[f'delta_{c}'].quantile(0.025)
    fs[f'delta_p97_5_{c}'] = g[f'delta_{c}'].quantile(0.975)
    fs[f'delta_sign_consistency_{c}'] = g[f'delta_{c}'].agg(lambda d: max((d > 0).mean(), (d < 0).mean()))
fs['detection_rate_any'] = g['sig_any'].mean()
fs['detection_rate_all'] = g['sig_all'].mean()
fs['interaction_rate'] = g['interaction_significant'].mean()
fs['equivalence_rate'] = g['equivalence_significant'].mean()
fs = fs.reset_index()
fs.to_csv(os.path.join(LAYER5_DIR, 'feature_stability.csv'), index=False)

agg = {'n_features': ('feature', 'size')}
for c in CONDITIONS:
    agg[f'detection_rate_{c}'] = (f'detection_rate_{c}', 'median')
    agg[f'delta_mean_{c}'] = (f'delta_mean_{c}', 'median')
    agg[f'delta_sd_{c}'] = (f'delta_sd_{c}', 'median')
family = fs.groupby(['domain', 'family']).agg(**agg).reset_index()
family.to_csv(os.path.join(LAYER5_DIR, 'family_stability.csv'), index=False)

print('Features significant in every seed:')
for c in CONDITIONS:
    print(f'  {c:5s}: {int((fs[f"detection_rate_{c}"] == 1).sum())} '
          f'(seed 42: {int(reference[f"sig_{c}"].sum())}; significant in at least one seed: '
          f'{int((fs[f"detection_rate_{c}"] > 0).sum())})')
print(f'  in all three conditions in every seed: {int((fs.detection_rate_all == 1).sum())} '
      f'(seed 42: {int(reference.sig_all.sum())})')
print("Median SD of Cliff's delta across seeds: "
      + ', '.join(f'{c} {fs[f"delta_sd_{c}"].median():.3f}' for c in CONDITIONS))
print(f'Seeds with a significant interaction: {int((seed_counts.n_interaction_fdr > 0).sum())} | '
      f'with a significant equivalence: {int((seed_counts.n_equivalent_fdr > 0).sum())}')
print('\nLargest families (median over features):')
print(family[family.n_features >= 10].sort_values('detection_rate_EC', ascending=False).round(2).to_string(index=False))

In [ ]:
# Significant features by domain, condition and seed
dom = pd.concat([r.groupby('domain')[[f'sig_{c}' for c in CONDITIONS] + ['sig_any', 'sig_all']].sum()
                 .reset_index().assign(seed=s) for s, r in runs.items()], ignore_index=True)
dom.to_csv(os.path.join(LAYER5_DIR, 'domain_counts.csv'), index=False)
cols = [f'sig_{c}' for c in CONDITIONS] + ['sig_any', 'sig_all']
summary = dom.groupby('domain')[cols].agg(['min', 'median', 'max']).astype(int)
summary.columns = [f'{a}_{b}' for a, b in summary.columns]
print('Significant features by domain across seeds (min, median, max):')
print(summary.to_string())

## 2. Amplitude scale (global RMS normalization)

With seed 42, the same windows and blocks as the main analysis. Features that do not depend on
the amplitude scale (relative power, ratios, asymmetry, IAF, spectral entropy, SEF95, CWT
coefficient of variation, entropies, fractal dimension, Hurst exponent, coherence and PLV) are
expected not to change; this is checked feature by feature (largest relative change across
subjects and conditions below 1e-6).

In [ ]:
rms_dir = os.path.join(LAYER5_DIR, 'rms_global')
run_many([(MAIN_SEED, 'rms_global', rms_dir)])
rms = read_run(rms_dir)

change = {}
for cond in CONDITIONS:
    a = pd.read_csv(os.path.join(RESULTS_DIR, f'features_{cond}.csv'), float_precision='round_trip').set_index('subject')[FEATURES]
    b = pd.read_csv(os.path.join(rms_dir, f'features_{cond}.csv'), float_precision='round_trip').set_index('subject').loc[a.index, FEATURES]
    change[cond] = ((b - a).abs() / a.abs().clip(lower=1e-12)).max()
rms_cmp = reference[['feature', 'domain', 'family']].copy()
rms_cmp['max_relative_change'] = pd.concat(change, axis=1).max(axis=1).reindex(rms_cmp.feature).to_numpy()
rms_cmp['scale_invariant'] = rms_cmp.max_relative_change < 1e-6
for c in CONDITIONS:
    rms_cmp[f'sig_main_{c}'] = reference[f'sig_{c}'].to_numpy()
    rms_cmp[f'sig_rms_{c}'] = rms.set_index('feature').loc[rms_cmp.feature, f'sig_{c}'].to_numpy()
    rms_cmp[f'delta_main_{c}'] = reference[f'delta_{c}'].to_numpy()
    rms_cmp[f'delta_rms_{c}'] = rms.set_index('feature').loc[rms_cmp.feature, f'delta_{c}'].to_numpy()
rms_cmp['status_main'] = reference.formal_condition_status.to_numpy()
rms_cmp['status_rms'] = rms.set_index('feature').loc[rms_cmp.feature, 'formal_condition_status'].to_numpy()
rms_cmp.to_csv(os.path.join(LAYER5_DIR, 'rms_feature_comparison.csv'), index=False)

rows = [{'run': 'main', **counts(reference)}, {'run': 'rms_global', **counts(rms)}]
rms_summary = pd.DataFrame(rows).set_index('run').T
sets = compare_sets(reference, rms).set_index('set')
for c in CONDITIONS:
    rho = spearmanr(rms_cmp[f'delta_main_{c}'], rms_cmp[f'delta_rms_{c}'], nan_policy='omit')[0]
    rms_summary.loc[f'spearman_delta_{c}', 'rms_global'] = rho
for s in sets.index:
    rms_summary.loc[f'jaccard_{s}', 'rms_global'] = sets.loc[s, 'jaccard']
rms_summary.loc['formal_status_agreement', 'rms_global'] = (rms_cmp.status_main == rms_cmp.status_rms).mean()
rms_summary.to_csv(os.path.join(LAYER5_DIR, 'rms_summary.csv'))
print(rms_summary.round(3).to_string())

inv = rms_cmp[rms_cmp.scale_invariant]
changed = (inv[[f'sig_main_{c}' for c in CONDITIONS]].to_numpy() != inv[[f'sig_rms_{c}' for c in CONDITIONS]].to_numpy()).any(axis=1)
print(f'\nScale-invariant features: {len(inv)} of {len(rms_cmp)}; significance changed in {int(changed.sum())} of them')
fam = rms_cmp.groupby('family').scale_invariant.mean()
print('Families that change with the amplitude scale:', sorted(fam[fam == 0].index))
print('Families partly invariant:', sorted(fam[(fam > 0) & (fam < 1)].index) or 'none')